# 12 – Joining Parcels and Assessor Attributes

This notebook joins the **clean parcel geometries** with the **clean assessor attributes**.

**Goals:**
- Load cleaned parcel layer (GeoDataFrame).
- Load cleaned assessor attribute table (DataFrame).
- Ensure parcel ID formats match.
- Perform a left join from parcels to assessor.
- Sanity-check join results (match counts, missing joins).
- Save a combined parcels + assessor layer for further analysis.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels"
ASSESSOR_DIR = DATA_DIR / "assessor"

clean_parcels_path = PARCELS_DIR / "processed/maricopa_parcels_cleaned.gpkg"    # from notebook 10
clean_assessor_path = ASSESSOR_DIR / "processed/R1170_SecMaster_2024_TAX_ROLL_clean.csv"  # from notebook 11

clean_parcels_path, clean_assessor_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/maricopa_parcels_cleaned.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/processed/R1170_SecMaster_2024_TAX_ROLL_clean.csv'))

In [2]:
parcels = gpd.read_file(clean_parcels_path)
assessor = pd.read_csv(clean_assessor_path)

print("Parcels:", len(parcels))
print("Assessor records:", len(assessor))

Parcels: 1736192
Assessor records: 1749565


In [3]:
parcels.columns

Index(['APN', 'Floor', 'LandSize', 'StartDate', 'Shp_Area', 'Shp_Length',
       'geometry'],
      dtype='object')

In [4]:
assessor.columns

Index(['PARCEL_NO', 'PROPERTYUSECODE', 'FULLCASHVALUE', 'LANDFULLCASHVALUE',
       'IMPRFULLCASHVALUE', 'LPVAMOUNT', 'LPVASSESSEDVALUE', 'SQFOOTAGE',
       'NUMBEROFUNITS', 'NEIGHBORHOODCODE', 'MARKETAREACODE'],
      dtype='object')

In [5]:
# join primary keys

PARCELS_ID_FIELD = "APN"
ASSESSOR_ID_FIELD = "PARCEL_NO"

PARCELS_ID_FIELD, ASSESSOR_ID_FIELD

('APN', 'PARCEL_NO')

In [6]:
# Convert both IDs to string, strip whitespace, make uppercase for consistency

parcels[PARCELS_ID_FIELD] = (
    parcels[PARCELS_ID_FIELD]
    .astype(str)
    .str.strip()
    .str.upper()
)

assessor[ASSESSOR_ID_FIELD] = (
    assessor[ASSESSOR_ID_FIELD]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [12]:
# Check if assessor parcel IDs are unique
assessor_id_counts = assessor[ASSESSOR_ID_FIELD].value_counts()
duplicates = (assessor_id_counts > 1).sum()
print("Assessor IDs with duplicates:", duplicates)

Assessor IDs with duplicates: 21


In [13]:
# Left join: keep all parcels, bring over assessor attributes where available

parcels_assessor = parcels.merge(
    assessor,
    how="left",
    left_on=PARCELS_ID_FIELD,
    right_on=ASSESSOR_ID_FIELD,
    suffixes=("", "_assessor")
)

print("Rows after join (should match parcels):", len(parcels_assessor))
parcels_assessor.head()

Rows after join (should match parcels): 1736192


,APN,Floor,LandSize,StartDate,Shp_Area,Shp_Length,geometry,PARCEL_NO,PROPERTYUSECODE,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,NEIGHBORHOODCODE,MARKETAREACODE
0,10101001C,1,7565.0,2008-09-22,7505.147490,350.125679,"MULTIPOLYGON Z (((581182.478 886654.38 0, 5811...",10101001C,5400.0,68700.0,68700.0,0.0,68700.0,11336.0,7565.0,NaN,1.0,6.0
1,10101010,1,195864.0,2008-09-22,195989.095940,1838.766520,"MULTIPOLYGON Z (((582322.095 889960.815 0, 582...",10101010,9780.0,8667760.0,1196500.0,7471260.0,7435591.0,1115339.0,195864.0,NaN,1.0,6.0
2,10101011,1,95491.0,2008-09-22,95501.799733,1248.957937,"MULTIPOLYGON Z (((581500.432 889688.576 0, 581...",10101011,9720.0,5190674.0,636700.0,4553974.0,3377292.0,506594.0,95491.0,NaN,1.0,6.0
3,10101012,1,66960.0,2008-09-22,66976.092596,1014.388578,"MULTIPOLYGON Z (((581428.706 889618.792 0, 581...",10101012,9705.0,553400.0,553400.0,0.0,305062.0,45759.0,66960.0,NaN,1.0,6.0
4,10101014,1,126838.0,2008-09-22,126838.044773,1424.568514,"MULTIPOLYGON Z (((581359.726 889391.699 0, 581...",10101014,9720.0,7074148.0,969800.0,6104348.0,2971142.0,445671.0,126838.0,NaN,1.0,6.0


In [14]:
# Count how many parcels did NOT find a match in assessor
no_match_count = parcels_assessor[ASSESSOR_ID_FIELD].isna().sum()
match_rate = 1 - no_match_count / len(parcels_assessor)

print("Parcels with no assessor match:", no_match_count)
print("Match rate:", round(match_rate * 100, 2), "%")

Parcels with no assessor match: 5116
Match rate: 99.71 %


## Join diagnostics

Explain what you found:

- What percentage of parcels successfully joined to assessor data?
- Are missing joins concentrated in any area or type of parcel?
- Are there obvious formatting or ID issues you still need to fix?

This explanation is important for the methods section of the project report.

In [16]:
joined_output_path = PARCELS_DIR / "processed/maricopa_parcels_with_assessor.gpkg"

parcels_assessor.to_file(joined_output_path, driver="GPKG")
print("Saved joined parcels+assessor layer to:", joined_output_path)

Saved joined parcels+assessor layer to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\processed\maricopa_parcels_with_assessor.gpkg
